# MPPI -- Full Evaluation (Validation & Test cases)

Chay MPPI lan luot tren tung case tu manifest CSV, dung --mode SANDBOX.
Tranh cac rang buoc cua VALIDATION/TEST mode (benchmark_eligible, git clean, metadrive commit).

| Split      | Cases | Tiers                                     |
|------------|-------|-------------------------------------------|
| Validation | 96    | Easy x24, Medium x24, Hard x24, Extreme x24 |
| Test       | 60    | Easy x15, Medium x15, Hard x15, Extreme x15 |

## Cell 1 -- Setup

In [12]:
import json, csv, subprocess, sys, time, pathlib, re
from typing import Optional, Dict, List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Project root
PROJECT_ROOT = pathlib.Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent:
    if (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT / "runs").exists():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

RUNS_DIR    = PROJECT_ROOT / "runs"
RESULTS_DIR = PROJECT_ROOT / "results" / "mppi_benchmark"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

MANIFEST_DIR = PROJECT_ROOT / "results" / "audits" / "evaluation_protocol"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RESULTS_DIR :", RESULTS_DIR)

PROJECT_ROOT: /home/nii/Documents/[code]sgu-2026-team-advanced-ai-subject/autonomous-driving-rl
RESULTS_DIR : /home/nii/Documents/[code]sgu-2026-team-advanced-ai-subject/autonomous-driving-rl/results/mppi_benchmark


## Cell 2 -- [edit] Cau hinh (chinh o day)

In [13]:
# "VALIDATION" = 96 cases | "TEST" = 60 cases | "BOTH" = ca hai
RUN_SPLIT = "VALIDATION"

# Config MPPI
MPPI_CONFIG_FILE = "mppi_v1.json"

# Agent seed
AGENT_SEED = 101

# Prefix cho run-id cua tung case
RUN_ID_PREFIX = "bm_mppi"

# True: bo qua case da co run roi
SKIP_EXISTING = True

# Gioi han so case chay (None = chay het)
# Dat 5 de thu truoc khi chay toan bo
MAX_CASES = None

# In xac nhan
config_path = PROJECT_ROOT / "configs" / "experiments" / MPPI_CONFIG_FILE
params = json.loads(config_path.read_text())["parameters"]
print(f"Split      : {RUN_SPLIT}")
print(f"Config     : {MPPI_CONFIG_FILE}")
print(f"  H={params['horizon']}, K={params['num_samples']}, lambda={params['temperature']}")
print(f"  target_speed={params['target_speed_m_s']} m/s = {params['target_speed_m_s']*3.6:.1f} km/h")
print(f"Max cases  : {MAX_CASES if MAX_CASES else 'ALL'}")

Split      : VALIDATION
Config     : mppi_v1.json
  H=20, K=512, lambda=0.5
  target_speed=16.0 m/s = 57.6 km/h
Max cases  : ALL


## Cell 3 -- Doc manifest va chay tung case

In [14]:
# --- Helpers ---

def patch_mppi_config(config_filename: str):
    target = (PROJECT_ROOT / "configs" / "experiments" / config_filename).as_posix()
    mppi_py = PROJECT_ROOT / "src" / "agents" / "MPPI.py"
    lines = mppi_py.read_text(encoding="utf-8").split("\n")
    new_lines = []
    for line in lines:
        if line.strip().startswith("CONFIG_PATH") and ("Path(__file__)" in line or 'pathlib.Path(r"' in line):
            new_lines.append(f'CONFIG_PATH = pathlib.Path(r"{target}")')
        else:
            new_lines.append(line)
    mppi_py.write_text("\n".join(new_lines), encoding="utf-8")


def restore_mppi_config():
    mppi_py = PROJECT_ROOT / "src" / "agents" / "MPPI.py"
    lines = mppi_py.read_text(encoding="utf-8").split("\n")
    new_lines = []
    patched = False
    for line in lines:
        if not patched and line.strip().startswith("CONFIG_PATH") and 'pathlib.Path(r"' in line:
            new_lines += [
                'CONFIG_PATH = (',
                '    Path(__file__).resolve().parents[2]',
                '    / "configs/experiments/mppi_v2_h30.json"',
                ')'
            ]
            patched = True
        else:
            new_lines.append(line)
    mppi_py.write_text("\n".join(new_lines), encoding="utf-8")


def make_run_id(split_prefix: str, row: Dict) -> str:
    safe_seq = re.sub(r'[^a-zA-Z0-9]', '', row['sequence'])[:8]
    return f"{RUN_ID_PREFIX}_{split_prefix}_{row['tier'][:2]}_{safe_seq}_g{row['geometry_generation_seed']}_e{row['environment_seed']}".lower()


def run_exists(run_id: str) -> Optional[pathlib.Path]:
    p = RUNS_DIR / run_id
    return p if (p.exists() and (p / "summary.json").exists()) else None


def run_one_case(row: Dict, run_id: str) -> Optional[pathlib.Path]:
    if SKIP_EXISTING:
        existing = run_exists(run_id)
        if existing:
            return existing

    cmd = [
        sys.executable, "-m", "src.launcher.cli", "run",
        "--mode", "SANDBOX",
        "--agent", "stage3_mppi",
        "--tier", row["tier"],
        "--sequence", row["sequence"],
        "--geom-seed", row["geometry_generation_seed"],
        "--env-seed",  row["environment_seed"],
        "--agent-seed", str(AGENT_SEED),
        "--wandb", "DISABLED",
        "--run-id", run_id,
    ]
    result = subprocess.run(cmd, cwd=str(PROJECT_ROOT), capture_output=True, text=True)
    if result.returncode != 0:
        return None
    return run_exists(run_id)


def load_manifest(split: str) -> List[Dict]:
    fname = "validation_case_manifest.csv" if split == "VALIDATION" else "test_case_manifest.csv"
    p = MANIFEST_DIR / fname
    with open(p, encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    # Sort theo protocol_order_index
    rows.sort(key=lambda r: int(r["protocol_order_index"]))
    return rows


def run_all_cases(split: str) -> pd.DataFrame:
    rows = load_manifest(split)
    if MAX_CASES:
        rows = rows[:MAX_CASES]
    prefix = "val" if split == "VALIDATION" else "test"

    total = len(rows)
    records = []
    t_start = time.time()

    for i, row in enumerate(rows, 1):
        run_id = make_run_id(prefix, row)
        existing = run_exists(run_id)
        status_prefix = "[SKIP]" if (SKIP_EXISTING and existing) else "[RUN] "

        t0 = time.time()
        rd = run_one_case(row, run_id)
        elapsed = time.time() - t0

        # Uoc tinh thoi gian con lai
        done_so_far = i
        avg_s = (time.time() - t_start) / done_so_far
        eta_min = avg_s * (total - done_so_far) / 60

        if rd:
            ep_df = pd.read_csv(rd / "episodes.csv")
            rc  = ep_df["final_route_completion"].iloc[0]
            ok  = ep_df["clean_success"].iloc[0]
            cr  = ep_df["crash_vehicle"].iloc[0]
            spd = ep_df["mean_speed_kmh"].iloc[0]
            ret = ep_df["episode_return"].iloc[0]
            records.append({
                "split": split, "run_id": run_id,
                "tier": row["tier"], "sequence": row["sequence"],
                "geom_seed": int(row["geometry_generation_seed"]),
                "env_seed":  int(row["environment_seed"]),
                "traffic_density": float(row["traffic_density"]),
                "final_route_completion": float(rc),
                "clean_success": bool(ok),
                "crash_vehicle": bool(cr),
                "out_of_road":   bool(ep_df["out_of_road"].iloc[0]),
                "timeout":       bool(ep_df["timeout"].iloc[0]),
                "mean_speed_kmh": float(spd),
                "episode_return": float(ret),
            })
            result_str = f"RC={rc:.2f} ok={int(ok)} cr={int(cr)} {elapsed:.0f}s ETA={eta_min:.0f}min"
        else:
            records.append({"split": split, "run_id": run_id,
                            "tier": row["tier"], "sequence": row["sequence"],
                            "geom_seed": int(row["geometry_generation_seed"]),
                            "env_seed":  int(row["environment_seed"]),
                            "final_route_completion": float("nan")})
            result_str = "FAILED"

        print(f"  [{i:3d}/{total}] {status_prefix} {row['tier']:7s} {row['sequence']:12s} g={row['geometry_generation_seed']:3s} e={row['environment_seed']:5s} -> {result_str}")

    return pd.DataFrame(records)

In [15]:
# Chay
patch_mppi_config(MPPI_CONFIG_FILE)

val_df = test_df = None
try:
    if RUN_SPLIT in ("VALIDATION", "BOTH"):
        print(f"\n=== VALIDATION (96 cases) ===")
        val_df = run_all_cases("VALIDATION")
        val_df.to_csv(RESULTS_DIR / "val_episodes.csv", index=False)
        print(f"\nSaved: {RESULTS_DIR / 'val_episodes.csv'}")

    if RUN_SPLIT in ("TEST", "BOTH"):
        print(f"\n=== TEST (60 cases) ===")
        test_df = run_all_cases("TEST")
        test_df.to_csv(RESULTS_DIR / "test_episodes.csv", index=False)
        print(f"\nSaved: {RESULTS_DIR / 'test_episodes.csv'}")
finally:
    restore_mppi_config()
    print("\nConfig restored.")


=== VALIDATION (96 cases) ===
  [  1/96] [RUN]  Extreme SCXOCrTYCS   g=19  e=5102  -> FAILED
  [  2/96] [RUN]  Medium  SCXCS        g=16  e=5101  -> FAILED
  [  3/96] [RUN]  Extreme SCTXORyCCS   g=18  e=5102  -> FAILED
  [  4/96] [RUN]  Hard    XTOCS        g=14  e=5101  -> FAILED
  [  5/96] [RUN]  Extreme CrXROSTR     g=10  e=5102  -> FAILED
  [  6/96] [RUN]  Hard    XTOCS        g=4   e=5102  -> FAILED
  [  7/96] [RUN]  Hard    XTOCS        g=9   e=5102  -> FAILED
  [  8/96] [RUN]  Hard    SCTXrCS      g=6   e=5101  -> FAILED
  [  9/96] [RUN]  Hard    SCTXrCS      g=14  e=5101  -> FAILED
  [ 10/96] [RUN]  Extreme SCTXORyCCS   g=18  e=5101  -> FAILED
  [ 11/96] [RUN]  Hard    SCTXrCS      g=9   e=5102  -> FAILED
  [ 12/96] [RUN]  Extreme SCXOCrTYCS   g=17  e=5102  -> FAILED
  [ 13/96] [RUN]  Extreme SCXOCrTYCS   g=17  e=5101  -> FAILED
  [ 14/96] [RUN]  Easy    SCS          g=6   e=5101  -> FAILED
  [ 15/96] [RUN]  Easy    SCSS         g=17  e=5101  -> FAILED
  [ 16/96] [RUN]  Easy  

## Cell 4 -- Load lai ket qua (neu chay lai notebook)

In [16]:
# Neu chua co tu Cell 3, load tu CSV da luu
if val_df is None and (RESULTS_DIR / "val_episodes.csv").exists():
    val_df = pd.read_csv(RESULTS_DIR / "val_episodes.csv")
    print(f"Loaded val_df: {len(val_df)} rows")

if test_df is None and (RESULTS_DIR / "test_episodes.csv").exists():
    test_df = pd.read_csv(RESULTS_DIR / "test_episodes.csv")
    print(f"Loaded test_df: {len(test_df)} rows")

# In tom tat nhanh
for label, df in [("VALIDATION", val_df), ("TEST", test_df)]:
    if df is None: continue
    valid = df.dropna(subset=["final_route_completion"])
    print(f"\n{label}: {len(valid)}/{len(df)} cases thanh cong")
    print(f"  RC mean    : {valid['final_route_completion'].mean():.3f}")
    print(f"  Success    : {valid['clean_success'].mean():.3f}")
    print(f"  Crash      : {valid['crash_vehicle'].mean():.3f}")
    print(f"  Out-of-road: {valid['out_of_road'].mean():.3f}")
    print(f"  Speed km/h : {valid['mean_speed_kmh'].mean():.1f}")
    print(f"  Return     : {valid['episode_return'].mean():.3f}")


VALIDATION: 0/96 cases thanh cong
  RC mean    : nan


KeyError: 'clean_success'

## Cell 5 -- Plot: RC / Success / Crash theo Tier

In [ ]:
def plot_by_tier(df: pd.DataFrame, split_label: str, save_path=None):
    tiers  = ["Easy", "Medium", "Hard", "Extreme"]
    cols   = ["final_route_completion", "clean_success", "crash_vehicle"]
    names  = ["Route Completion", "Success Rate", "Crash Rate"]
    colors = ["#4e79a7", "#59a14f", "#f28e2b", "#e15759"]

    df = df.dropna(subset=["final_route_completion"])
    fig, axes = plt.subplots(1, 3, figsize=(13, 4))

    for ax, col, name in zip(axes, cols, names):
        vals = [df[df["tier"]==t][col].astype(float).mean() for t in tiers]
        errs = [df[df["tier"]==t][col].astype(float).std()  for t in tiers]
        bars = ax.bar(tiers, vals, yerr=errs, capsize=4,
                      color=colors, alpha=0.85, error_kw={"lw": 1.5})
        ax.set_title(name)
        ax.set_ylim(0, 1.15)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, v + 0.04,
                    f"{v:.2f}", ha="center", fontsize=9)

    overall_rc = df["final_route_completion"].mean()
    overall_ok = df["clean_success"].mean()
    overall_cr = df["crash_vehicle"].mean()
    fig.suptitle(
        f"MPPI {split_label} ({len(df)} cases) | "
        f"RC={overall_rc:.3f} Success={overall_ok:.3f} Crash={overall_cr:.3f}",
        fontweight="bold"
    )
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()


if val_df is not None:
    plot_by_tier(val_df, "Validation", RESULTS_DIR / "plot1_val_by_tier.png")
if test_df is not None:
    plot_by_tier(test_df, "Test", RESULTS_DIR / "plot1_test_by_tier.png")

## Cell 6 -- Plot: Phan phoi Route Completion

In [ ]:
def plot_rc_hist(df: pd.DataFrame, split_label: str, save_path=None):
    tiers  = ["Easy", "Medium", "Hard", "Extreme"]
    colors = ["#4e79a7", "#59a14f", "#f28e2b", "#e15759"]
    df = df.dropna(subset=["final_route_completion"])

    fig, axes = plt.subplots(1, 4, figsize=(14, 3.5), sharey=True)
    for ax, tier, color in zip(axes, tiers, colors):
        sub = df[df["tier"]==tier]["final_route_completion"].astype(float)
        ax.hist(sub, bins=15, range=(0, 1), color=color, alpha=0.8)
        ax.axvline(sub.mean(), color="black", ls="--", lw=1.5,
                   label=f"mean={sub.mean():.2f}")
        ax.set_title(f"{tier} (n={len(sub)})")
        ax.set_xlabel("Route Completion")
        ax.legend(fontsize=8)
    axes[0].set_ylabel("Count")
    fig.suptitle(f"MPPI {split_label}: Phan phoi Route Completion", fontweight="bold")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()


if val_df is not None:
    plot_rc_hist(val_df, "Validation", RESULTS_DIR / "plot2_val_rc_hist.png")
if test_df is not None:
    plot_rc_hist(test_df, "Test", RESULTS_DIR / "plot2_test_rc_hist.png")

## Cell 7 -- Plot: Stacked bar ket qua episode

In [ ]:
def plot_outcomes(df: pd.DataFrame, split_label: str, save_path=None):
    tiers = ["Easy", "Medium", "Hard", "Extreme"]
    df = df.dropna(subset=["final_route_completion"])

    outcome_map = [
        ("clean_success",  "Success",     "#2ca02c"),
        ("crash_vehicle",  "Crash",        "#d62728"),
        ("out_of_road",    "Out-of-Road",  "#ff7f0e"),
        ("timeout",        "Timeout",      "#9467bd"),
    ]

    fig, ax = plt.subplots(figsize=(9, 4))
    x = np.arange(len(tiers))
    bottom = np.zeros(len(tiers))

    for col, name, color in outcome_map:
        if col not in df.columns: continue
        vals = np.array([df[df["tier"]==t][col].astype(float).mean() for t in tiers])
        ax.bar(x, vals, bottom=bottom, label=name, color=color, alpha=0.85)
        bottom += vals

    ax.set_xticks(x)
    ax.set_xticklabels(tiers)
    ax.set_ylabel("Rate")
    ax.set_ylim(0, 1.05)
    ax.set_title(f"MPPI {split_label}: Ket qua episode theo Tier")
    ax.legend(loc="upper right", fontsize=9)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()


if val_df is not None:
    plot_outcomes(val_df, "Validation", RESULTS_DIR / "plot3_val_outcomes.png")
if test_df is not None:
    plot_outcomes(test_df, "Test", RESULTS_DIR / "plot3_test_outcomes.png")

## Cell 8 -- Plot: Val vs Test (neu co ca hai)

In [ ]:
def plot_val_vs_test(val: Optional[pd.DataFrame], test: Optional[pd.DataFrame], save_path=None):
    if val is None and test is None:
        print("Can it nhat 1 split.")
        return

    tiers   = ["Easy", "Medium", "Hard", "Extreme"]
    metrics = ["final_route_completion", "clean_success", "crash_vehicle"]
    names   = ["Route Completion", "Success Rate", "Crash Rate"]

    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    x = np.arange(len(tiers))
    w = 0.35

    for ax, col, name in zip(axes, metrics, names):
        if val is not None:
            v_df = val.dropna(subset=["final_route_completion"])
            vals_v = [v_df[v_df["tier"]==t][col].astype(float).mean() for t in tiers]
            ax.bar(x - w/2, vals_v, w, label="Validation", color="steelblue", alpha=0.85)
        if test is not None:
            t_df = test.dropna(subset=["final_route_completion"])
            vals_t = [t_df[t_df["tier"]==t][col].astype(float).mean() for t in tiers]
            ax.bar(x + w/2, vals_t, w, label="Test", color="tomato", alpha=0.85)
        ax.set_xticks(x)
        ax.set_xticklabels(tiers)
        ax.set_title(name)
        ax.set_ylim(0, 1.15)
        ax.legend(fontsize=9)

    fig.suptitle("MPPI -- Validation vs Test", fontweight="bold")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()


plot_val_vs_test(val_df, test_df, RESULTS_DIR / "plot4_val_vs_test.png")

## Cell 9 -- Bang tong hop + luu CSV

In [ ]:
rows = []
for label, df in [("Validation", val_df), ("Test", test_df)]:
    if df is None: continue
    df = df.dropna(subset=["final_route_completion"])
    for tier in ["Easy", "Medium", "Hard", "Extreme", "ALL"]:
        sub = df if tier == "ALL" else df[df["tier"] == tier]
        if len(sub) == 0: continue
        rows.append({
            "Split":   label,
            "Tier":    tier,
            "N":       len(sub),
            "RC mean": round(sub["final_route_completion"].astype(float).mean(), 3),
            "RC std":  round(sub["final_route_completion"].astype(float).std(),  3),
            "Success": round(sub["clean_success"].astype(float).mean(),  3),
            "Crash":   round(sub["crash_vehicle"].astype(float).mean(),  3),
            "OOR":     round(sub["out_of_road"].astype(float).mean(),    3),
            "Speed":   round(sub["mean_speed_kmh"].astype(float).mean(), 1),
            "Return":  round(sub["episode_return"].astype(float).mean(), 3),
        })

table = pd.DataFrame(rows)
print(table.to_string(index=False))

table.to_csv(RESULTS_DIR / "benchmark_summary.csv", index=False)
print(f"\nSaved: {RESULTS_DIR / 'benchmark_summary.csv'}")
for f in sorted(RESULTS_DIR.glob("*.png")):
    print(f"  {f.name}")